In [1]:
import tensorflow as tf
import keras
from sklearn.model_selection import train_test_split

# 0) 랜덤 시드 고정 및 연산 결정성 설정 (스크린샷 최상단 요구사항)
keras.utils.set_random_seed(42)
tf.config.experimental.enable_op_determinism()

In [2]:
# 1) Fashion MNIST 로딩 + 전처리 (Input(28,28) 방식 - reshape 없이)
(train_input, train_target), (test_input, test_target) = keras.datasets.fashion_mnist.load_data()

In [3]:
# 0~1 사이로 스케일링 (reshape은 하지 않음)
train_scaled = train_input / 255.0

In [4]:
# 검증 세트 분리 (발급받으신 주피터 파일 기준 test_size=0.2 적용)
train_scaled, val_scaled, train_target, val_target = train_test_split(
    train_scaled, train_target, test_size=0.2, random_state=42
)

In [5]:
# 2) Flatten + Dense(100, relu) + Dense(10, softmax) 모델 선언
model = keras.Sequential([
    # reshape 대신 입력층에 2D 구조를 그대로 받고, 1D로 펼쳐주는 Flatten 레이어 적용
    keras.layers.Flatten(input_shape=(28, 28), name='flatten'),
    keras.layers.Dense(units=100, activation='relu', name='dense_hidden'),
    keras.layers.Dense(units=10, activation='softmax', name='dense_output')
], name='fashion_mnist_flatten_model')

c:\Users\RookieFit\p1-data\c3-deep-learning\.venv\Lib\site-packages\keras\src\layers\reshaping\flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [6]:
# 3) compile(optimizer='adam', loss='sparse_categorical_crossentropy')
# 평가지표(metrics)에 'accuracy'를 추가해야 evaluate 시 val_accuracy를 확인할 수 있습니다.
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [7]:
# 모델 요약 확인 (Total params: 79,510 및 Flatten params: 0 요구사항 만족 여부 확인)
model.summary()

Model: "fashion_mnist_flatten_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_hidden (Dense)            │ (None, 100)            │        78,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_output (Dense)            │ (None, 10)             │         1,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 79,510 (310.59 KB)

 Trainable params: 79,510 (310.59 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
# 4) fit(epochs=5)
model.fit(train_scaled, train_target, epochs=6)

Epoch 1/6
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8954 - loss: 0.2866
Epoch 2/6
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.8994 - loss: 0.2732
Epoch 3/6
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9040 - loss: 0.2609
Epoch 4/6
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9091 - loss: 0.2504
Epoch 5/6
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9126 - loss: 0.2409
Epoch 6/6
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9158 - loss: 0.2321


In [11]:
# 5) evaluate → val_accuracy 확인
print("\n--- 검증 세트(Validation Set) 평가 결과 ---")
val_loss, val_acc = model.evaluate(val_scaled, val_target)


--- 검증 세트(Validation Set) 평가 결과 ---
375/375 ━━━━━━━━━━━━━━━━━━━━ 0s 925us/step - accuracy: 0.8813 - loss: 0.3456


In [14]:
import tensorflow as tf
import keras
from sklearn.model_selection import train_test_split
import numpy as np

# 데이터 로드 및 전처리 함수 (재사용성 위함)
def load_and_preprocess_mnist():
    (train_input, train_target), (test_input, test_target) = keras.datasets.fashion_mnist.load_data()
    train_scaled = train_input / 255.0
    return train_test_split(train_scaled, train_target, test_size=0.2, random_state=42)

# [APPLIED] 모델 생성 함수
def make_model():
    m = keras.Sequential()
    m.add(keras.layers.Input(shape=(28, 28)))
    m.add(keras.layers.Flatten())
    m.add(keras.layers.Dense(100, activation='relu'))
    m.add(keras.layers.Dense(10, activation='softmax'))
    return m

# 데이터 준비
train_scaled, val_scaled, train_target, val_target = load_and_preprocess_mnist()

print("=== [APPLIED] SGD vs Adam 비교 시작 ===")

# 스크린샷 딕셔너리 요구사항 그대로 반영
optimizers_dict = {
    'SGD (기본)': keras.optimizers.SGD(),
    'SGD (lr=0.1)': keras.optimizers.SGD(learning_rate=0.1),
    'Adam': keras.optimizers.Adam()
}

for opt_name, opt in optimizers_dict.items():
    # 실험의 재현성을 위해 루프 돌 때마다 시드 초기화
    keras.utils.set_random_seed(42)
    tf.config.experimental.enable_op_determinism()
    
    model = make_model()
    model.compile(optimizer=opt, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    
    # 훈련 (스크린샷 요구사항대로 verbose=0으로 화면 무출력)
    model.fit(train_scaled, train_target, epochs=5, verbose=0)
    
    # 평가
    _, val_acc = model.evaluate(val_scaled, val_target, verbose=0)
    print(f"{opt_name}: {val_acc:.4f}")

print("\n=== [CHALLENGE] Part 1: Adam 학습률(lr) 조정 실험 ===")

# 학습률 후보 리스트
lr_list = [0.1, 0.01, 0.001, 0.0001]

for lr in lr_list:
    keras.utils.set_random_seed(42)
    tf.config.experimental.enable_op_determinism()
    
    model = make_model()
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), 
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    
    model.fit(train_scaled, train_target, epochs=5, verbose=0)
    _, val_acc = model.evaluate(val_scaled, val_target, verbose=0)
    print(f"Adam (lr={lr}): {val_acc:.4f}")

print("\n=== [CHALLENGE] Part 2: CIFAR-10 Dense 한계 실험 ===")

# CIFAR-10 데이터 세트 로드 (32x32 RGB 컬러 이미지)
(cifar_train, cifar_train_target), (cifar_test, cifar_test_target) = keras.datasets.cifar10.load_data()
cifar_train_scaled = cifar_train / 255.0
c_train_scaled, c_val_scaled, c_train_target, c_val_target = train_test_split(
    cifar_train_scaled, cifar_train_target, test_size=0.2, random_state=42
)

# 스크린샷 구조 그대로 설계된 CIFAR-10 모델
model_cifar = keras.Sequential()
model_cifar.add(keras.layers.Input(shape=(32, 32, 3)))
model_cifar.add(keras.layers.Flatten()) # 32 * 32 * 3 = 3,072 차원으로 플래튼
model_cifar.add(keras.layers.Dense(100, activation='relu'))
model_cifar.add(keras.layers.Dense(10, activation='softmax'))

# 가장 좋은 성능을 냈던 기본 Adam으로 컴파일
model_cifar.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# 학습 및 평가
model_cifar.fit(c_train_scaled, c_train_target, epochs=5, verbose=0)
_, cifar_val_acc = model_cifar.evaluate(c_val_scaled, c_val_target, verbose=0)
print(f"CIFAR-10 Dense 모델 최종 val_accuracy: {cifar_val_acc:.4f}")

=== [APPLIED] SGD vs Adam 비교 시작 ===
SGD (기본): 0.8478
SGD (lr=0.1): 0.8575
Adam: 0.8710

=== [CHALLENGE] Part 1: Adam 학습률(lr) 조정 실험 ===
Adam (lr=0.1): 0.2939
Adam (lr=0.01): 0.8519
Adam (lr=0.001): 0.8710
Adam (lr=0.0001): 0.8573

=== [CHALLENGE] Part 2: CIFAR-10 Dense 한계 실험 ===
CIFAR-10 Dense 모델 최종 val_accuracy: 0.3815
